# Day 02：运算符与字符串（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 1 周**：第 1 周：Python 核心语法与数据结构基石  
> **GitLab 工单**：`STARMIND-002` ｜ **开发分支**：`feat/day02-text-cleaner-prompt`  
> **配套 HTML5 课件**：`courseware/day02/index.html` ｜ **独立版本快照**：`versions/day02/`

---

## 🎙️ 老陈开场旁白

早啊！刚开完早会，林产品在飞书里发了一堆真实客服工单截图——全是用户从微信和网页复制过来的脏数据：前后一堆全角空格、莫名其妙的换行符、夹杂着广告链接，还有大小写乱写的订单号。记住：大模型吃进去垃圾，吐出来的就是更贵的垃圾（Garbage In, Garbage Out），还会白白烧掉你几倍的 Token 费！今天 STARMIND-002 咱们吃透 Python 运算符和字符串操作，亲手写一个企业级「文本清洗工具」和「f-string Prompt 模板引擎」，给大模型喂最干净的粮食。

### 承上启下
- **昨日基线**：昨天咱们搭好了 Python 3.11 + 国内镜像源环境，完成了 STARMIND-001 员工数字名片工具，4 个 pytest 用例全部跑通。
- **明日预告**：明天（Day 3）咱们引入条件分支与循环流程控制，把今天单条清洗升级成可交互的命令行菜单与批量路由分发器。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day02'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-002）

客服系统每天产生数万条用户反馈文本，直接送入大模型分析会导致 Token 浪费高达 25%，且格式杂乱影响模型抽取准确度。

**今日迭代目标：**
- 掌握算术、比较、逻辑与成员运算符（in / not in）及短路求值特性。
- 熟练运用字符串切片、strip/replace/split/join/lower/upper 等核心方法。
- 使用 f-string 构建结构化 Prompt 模板，并实现可复用的文本清洗函数。

**验收标准：**
- 文本清洗器能清除首尾空白、折叠连续空格、统一全角空格并标准化订单号前缀。
- Prompt 模板能动态注入角色、用户问题、字数限制并计算估算 Token 数。
- 所有边界用例（空字符串、超长文本截断、敏感词命中检测）通过 pytest。

---

## 📚 课堂核心技术笔记

### 1. 1. 运算符与短路求值

算术运算中 `/` 永远返回浮点数，`//` 是向下取整除法，`%` 取余，`**` 幂运算。逻辑运算符 `and` / `or` 具备短路特性：`text and text.strip()` 会在 `text` 为空字符串或 `None` 时直接短路返回，不会触发 `.strip()` 报错。成员运算符 `in` / `not in` 在敏感词检测、关键词匹配中极其常用。

> **⚠️ 老陈避坑**：注意 `==` 判断值相等，而 `is` 判断内存地址是否为同一个对象！判断 `None` 用 `is None`，比较字符串或大整数永远用 `==`。

### 2. 2. 字符串不可变性与高频方法

Python 的 `str` 是不可变对象，调用 `s.strip()` 或 `s.replace('a', 'b')` 不会原地修改 `s`，而是返回一个新字符串，必须重新赋值！折叠任意连续空白（空格、换行、制表符）的最优雅写法是 `' '.join(text.split())`，一行搞定。

> **⚠️ 老陈避坑**：初学者最常犯的错：写了一行 `text.replace(' ', '')` 却没写 `text = text.replace(' ', '')`，结果发现字符串纹丝不动！

### 3. 3. f-string 与 Prompt 工程模板

f-string 支持直接嵌入表达式、格式化精度（`{val:.2f}`）、对齐填充（`{name:<10}`）以及多行三引号模板。在大模型开发里，用三引号 f-string 搭配明确分隔符（如 `###` 或 XML 标签 `<user_query>`）是防提示词混乱的标准姿势。

> **⚠️ 老陈避坑**：如果在 f-string 里要输出 JSON 的真实花括号 `{}`，必须双写转义为 `{{` 和 `}}`，否则 Python 会把它当变量解析报错。


## 💻 实操步骤 1：步骤一：编写客服工单文本清洗器

- **目标文件**：`versions/day02/src/day02_text_cleaner.py`
- **老陈代码旁白**：先解决脏文本问题。咱们把全角空格替换、多余换行折叠、订单号大写标准化和安全截断都封进 `clean_ticket_text` 函数。


In [ ]:
"""
STARMIND-002 模块一：企业客服工单文本清洗器。
演示字符串不可变性、核心方法（replace/split/join/upper）、切片截断与成员运算符。
"""

# 定义常见的违禁/高风险测试词（演示成员运算符 in）
BLOCKED_KEYWORDS = ("刷单", "代开发票", "木马链接")


def clean_ticket_text(raw_text: str, max_length: int = 200) -> str:
    """
    清洗客服工单原始文本：
    1. 将中文全角空格（\\u3000）替换为普通空格；
    2. 利用 split() + join() 折叠所有连续空格、制表符和多余换行；
    3. 统一把小写订单前缀 'ord-' 标准化为 'ORD-'；
    4. 若超过 max_length，使用切片截断并追加省略号。
    """
    # 防御 None 或空字符串输入
    if not raw_text:
        return ""

    # 第 1 步：替换全角空格与不间断空格
    normalized = raw_text.replace("\u3000", " ").replace("\xa0", " ")

    # 第 2 步：split() 不传参会按任意连续空白切分，再用单个空格拼回
    compact = " ".join(normalized.split())

    # 第 3 步：标准化常见小写订单号前缀
    standardized = compact.replace("ord-", "ORD-").replace("Ord-", "ORD-")

    # 第 4 步：使用切片进行长度控制，防止超长文本打爆 Token 预算
    if len(standardized) > max_length:
        # 预留 3 个字符给 '...'
        cutoff = max(0, max_length - 3)
        return standardized[:cutoff] + "..."

    return standardized


def has_blocked_content(text: str) -> bool:
    """检查清洗后的文本是否命中风险关键词。"""
    # 使用成员运算符 in 结合 any 表达式判断
    return any(keyword in text for keyword in BLOCKED_KEYWORDS)


if __name__ == "__main__":
    dirty = "  \u3000你好！！\n\n  我的订单  ord-20260501   一直没发货，   麻烦查一下\t谢谢！  "
    cleaned = clean_ticket_text(dirty, max_length=60)
    print(f"原始长度: {len(dirty)} -> 清洗后长度: {len(cleaned)}")
    print(f"清洗结果: {cleaned}")
    print(f"风险拦截: {has_blocked_content(cleaned)}")


### 🔍 步骤 1 代码深度解析

1. `raw_text.replace('\u3000', ' ')` 专门用来对付从微信、Word 复制过来的中文全角空格。
2. `' '.join(normalized.split())` 是 Python 老手最爱的文本压缩技巧，能同时干掉首尾空格、连续空格、`\n` 和 `\t`。
3. 切片 `standardized[:cutoff]` 即使 `cutoff` 超出字符串长度也不会抛出 IndexError，非常安全。

**预期运行输出参考：**
```text
原始长度: 50 -> 清洗后长度: 36
清洗结果: 你好！！ 我的订单 ORD-20260501 一直没发货， 麻烦查一下 谢谢！
风险拦截: False
```


## 💻 实操步骤 2：步骤二：打造 f-string Prompt 模板渲染与 Token 估算引擎

- **目标文件**：`versions/day02/src/day02_prompt_template.py`
- **老陈代码旁白**：文本洗干净了，接下来咱们用 f-string 把角色设定、业务规则、清洗后的用户提问组装成一份专业 Prompt，顺便用算术运算符估算一下大概消耗多少 Token。


In [ ]:
"""
STARMIND-002 模块二：f-string Prompt 模板引擎与 Token 粗估工具。
演示多行 f-string、转义花括号、算术运算符（整除 // 与向上取整技巧）。
"""

from day02_text_cleaner import clean_ticket_text, has_blocked_content


def estimate_tokens_rough(text: str) -> int:
    """
    粗略估算中英混合文本的 Token 数量（经验值：约每 2 个字符折算 1.5 个 Token）。
    使用整数运算避免浮点误差，并保证非空文本至少返回 1。
    """
    if not text:
        return 0
    # 向上取整技巧：(长度 * 3 + 3) // 4 等价于 ceil(len * 0.75)
    return max(1, (len(text) * 3 + 3) // 4)


def render_support_prompt(
    user_query: str,
    customer_tier: str = "VIP",
    max_reply_words: int = 120,
) -> str:
    """将清洗后的用户问题组装为星辰智脑标准客服 Prompt。"""
    cleaned_query = clean_ticket_text(user_query, max_length=300)

    # 若触发安全词拦截，直接返回拒答提示模板
    if has_blocked_content(cleaned_query):
        return "[BLOCKED] 检测到高风险违规内容，已拦截，不送入大模型。"

    # 使用多行 f-string 与 XML 标签清晰隔离用户输入，降低提示词注入风险
    # 注意：JSON 示例中的花括号必须双写 {{ }} 进行转义
    prompt = f"""你是「杭州星辰数科」官方智能客服助手，专业、克制且高效。
当前服务客户等级：{customer_tier.upper()}
回复字数要求：不超过 {max_reply_words} 字，必须输出如下 JSON 格式：
{{"intent": "物流查询|退款售后|其他", "reply": "你的回复内容"}}

请严格基于以下标签内的用户原始提问作答：
<user_query>
{cleaned_query}
</user_query>"""
    return prompt


if __name__ == "__main__":
    sample_q = "  \n 麻烦问下 ord-99812 啥时候能送到杭州滨江？\u3000急用！ "
    final_prompt = render_support_prompt(sample_q, customer_tier="svip", max_reply_words=80)
    est_tokens = estimate_tokens_rough(final_prompt)
    print(final_prompt)
    print(f"\n--- 预估输入 Token 数: 约 {est_tokens} Tokens ---")


### 🔍 步骤 2 代码深度解析

1. `(len(text) * 3 + 3) // 4` 是工程中常用的纯整数向上取整写法，无需引入 math 库就能算出 `ceil(n * 0.75)`。
2. Prompt 中用 `{{` 和 `}}` 转义了 JSON 模板的花括号，同时用 `<user_query>` 标签包裹用户输入，为 Day 18 的提示词注入防御埋下伏笔。

**预期运行输出参考：**
```text
你是「杭州星辰数科」官方智能客服助手，专业、克制且高效。
当前服务客户等级：SVIP
回复字数要求：不超过 80 字，必须输出如下 JSON 格式：
{"intent": "物流查询|退款售后|其他", "reply": "你的回复内容"}

请严格基于以下标签内的用户原始提问作答：
<user_query>
麻烦问下 ORD-99812 啥时候能送到杭州滨江？ 急用！
</user_query>

--- 预估输入 Token 数: 约 126 Tokens ---
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day02/tests/test_day02.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-002 自动化单元测试。"""

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day02_text_cleaner import clean_ticket_text, has_blocked_content
from day02_prompt_template import estimate_tokens_rough, render_support_prompt


def test_clean_ticket_text_whitespace_and_order_id():
    raw = " \u3000订单\t ord-10086 \n\n  未发货  "
    assert clean_ticket_text(raw) == "订单 ORD-10086 未发货"
    assert clean_ticket_text("") == ""


def test_clean_ticket_text_truncation():
    long_text = "星辰智脑" * 20  # 80 字符
    result = clean_ticket_text(long_text, max_length=15)
    assert len(result) == 15
    assert result.endswith("...")


def test_blocked_content_detection():
    assert has_blocked_content("你好请问支持刷单吗") is True
    assert has_blocked_content("请问发票怎么开") is False


def test_render_support_prompt_and_token_estimate():
    prompt = render_support_prompt("  查一下 ord-777  ", customer_tier="vip", max_reply_words=100)
    assert "VIP" in prompt
    assert "ORD-777" in prompt
    assert '{"intent":' in prompt
    assert estimate_tokens_rough(prompt) > 20
    assert render_support_prompt("点这个木马链接领取") .startswith("[BLOCKED]")

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：商品标题去噪与截断

**题目要求：**
电商运营上传的商品标题常带 `【爆款】`、`[限时]` 等营销前缀和多余空格。编写函数 `normalize_title(title: str, limit: int = 20) -> str`，去除 `【爆款】`、`[限时]` 并折叠空白，超长则切片截断加 `...`。

**思路提示：** 链式调用 `.replace()` 后再用 `' '.join(s.split())`，最后判断长度切片。


In [ ]:
"""Day 2 课后题 1：商品标题清洗与截断。"""


def normalize_title(title: str, limit: int = 20) -> str:
    """移除营销标签、压缩空白并按字数限制截断标题。"""
    cleaned = title.replace("【爆款】", "").replace("[限时]", "")
    cleaned = " ".join(cleaned.split())
    if len(cleaned) > limit:
        return cleaned[: max(0, limit - 3)] + "..."
    return cleaned


if __name__ == "__main__":
    raw = "  【爆款】  [限时]  星辰智脑   企业级AI知识库私有化部署套件  "
    print(normalize_title(raw, limit=16))


**答案解析与评分标准：**

考查字符串链式替换与切片边界计算。注意先去标签再统计长度，顺序不能反。


### 作业 2（企业实战）：企业实战：对话日志快速摘要统计器

**题目要求：**
给定一段包含多行对话的字符串（每行格式为 `USER: ...` 或 `AI: ...`），利用字符串方法统计总字符数、是否包含关键字 `'退款'`、以及 `USER:` 出现的次数，返回格式化报告。

**思路提示：** 使用 `str.count('USER:')`、`len()` 和成员运算符 `in`。


In [ ]:
"""Day 2 课后题 2：对话日志摘要统计器。"""


def summarize_chat_log(log_text: str) -> str:
    """统计对话日志的基础指标并返回摘要。"""
    cleaned = log_text.strip()
    user_turns = cleaned.count("USER:")
    ai_turns = cleaned.count("AI:")
    has_refund = "退款" in cleaned
    return (
        f"日志总字数: {len(cleaned)} | 用户轮次: {user_turns} | "
        f"AI 轮次: {ai_turns} | 涉及退款: {'是' if has_refund else '否'}"
    )


if __name__ == "__main__":
    sample_log = "USER: 你好，我想申请退款\nAI: 请提供您的订单号\nUSER: ORD-1001"
    print(summarize_chat_log(sample_log))


**答案解析与评分标准：**

利用 `str.count()` 无需写循环就能快速统计特征子串出现频次，适合日志初筛场景。


### 作业 3（架构进阶）：架构进阶：带变量校验的安全 Prompt 渲染器

**题目要求：**
业务同学经常手写 Prompt 模板如 `'你好 {name}，欢迎使用 {product}'`，但如果漏传变量会导致线上崩溃。请编写 `safe_render(template: str, **kwargs) -> str`，在渲染前检查必要占位符是否存在，若缺失则给出友好错误提示而不是直接崩掉。

**思路提示：** 捕获 `KeyError` 或提前检查关键字段；这里可以先用条件判断或 `template.format_map()` 配合默认占位。


In [ ]:
"""Day 2 课后题 3：防崩溃的 Prompt 模板安全渲染器。"""


class SafeDict(dict):
    """当模板缺少某个键时，保留原始占位符而不是抛出 KeyError。"""

    def __missing__(self, key: str) -> str:
        return f"<MISSING:{key}>"


def safe_render(template: str, variables: dict) -> str:
    """安全渲染模板，缺失字段自动标记为 <MISSING:key>。"""
    return template.format_map(SafeDict(variables))


if __name__ == "__main__":
    tpl = "你好 {user}，你的 {plan} 套餐剩余 {quota} Tokens。"
    print(safe_render(tpl, {"user": "陈小新", "plan": "企业版"}))


**答案解析与评分标准：**

利用 `str.format_map()` 与自定义字典缺失处理，既保证生产环境不因运营少配一个字段而 500 报错，又能在监控日志里一眼看出 `<MISSING:quota>`。


---

## 🎯 老陈收尾复盘

今天写完文本清洗和 Prompt 模板，你应该能感受到：大模型应用开发里至少 40% 的脏活累活本质上都是字符串处理。记住切片左闭右开、字符串不可变、拼接大文本优先用 `''.join()` 这三条铁律。晚上把三道题练熟，明天咱们学流程控制！
